# 01 — Reproducing TSCG compression

Independent Python reproduction of the TSCG paper ([arXiv:2605.04107](https://arxiv.org/abs/2605.04107)).

We sweep 3 catalogs × 3 profiles × 3 models, then plot the headline figure with both TSCG-reported and tiktoken-independent token savings.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

from tscg_bench.bench import run_benchmark, write_csv
from tscg_bench.catalog import list_catalogs, load_catalog
import pandas as pd

[p.name for p in list_catalogs()]

## Peek the small catalog

In [ ]:
catalog = load_catalog('small_5tools')
[t['function']['name'] for t in catalog]

## Run the 27-cell sweep

In [ ]:
rows = run_benchmark(
    catalogs=['small_5tools', 'mid_20tools', 'mcp_filesystem'],
    profiles=('conservative', 'balanced', 'aggressive'),
    models=('gpt-4', 'claude-sonnet', 'phi-4'),
)
df = pd.DataFrame([r.__dict__ for r in rows])
df['applied_principles'] = df['applied_principles'].apply(lambda t: ','.join(t))
df.head()

## Headline chart: TSCG-reported vs tiktoken-independent

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

focus = df[(df['profile']=='balanced') & (df['model']=='gpt-4')].sort_values('n_tools')
labels = [f"{r.catalog_name}\n({r.n_tools} tools)" for r in focus.itertuples()]
x = np.arange(len(labels))
width = 0.38

fig, ax = plt.subplots(figsize=(9, 5), dpi=120)
ax.bar(x - width/2, focus['savings_pct_tscg'], width, label='TSCG self-reported', color='#2563eb')
ax.bar(x + width/2, focus['savings_pct_tiktoken'], width, label='tiktoken independent', color='#dc2626')
ax.axhline(51, ls='--', lw=1.3, color='#16a34a', label='Theorem 3.1 floor (>=51%)')
ax.set_ylabel('Token savings (%)')
ax.set_title('TSCG paper vs independent reproduction (profile=balanced, model=gpt-4)')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 80); ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Aggregate stats

In [ ]:
print(f"TSCG savings range:     {df.savings_pct_tscg.min():.1f}% - {df.savings_pct_tscg.max():.1f}%")
print(f"tiktoken savings range: {df.savings_pct_tiktoken.min():.1f}% - {df.savings_pct_tiktoken.max():.1f}%")
print(f"Mean gap:               {(df.savings_pct_tscg - df.savings_pct_tiktoken).mean():.1f}pp")
print(f"Theorem 3.1 hits:       {df.meets_theorem_3_1.sum()}/{len(df)}")
print(f"Mean compile time:      {df.elapsed_ms.mean():.2f} ms")